In [ ]:
# ==============================================================================
# 🚀 ALPHA 450M — TENSORFLOW TPU v5e-8 MASTER TRAINING PIPELINE
#
# Pretraining Unrealistic Alpha (450M parameters) on Kaggle TPU v5e-8.
#
# Specifications:
# - Hardware: Kaggle TPU v5e-8 (8 Cores, 128 GB HBM, mixed_bfloat16)
# - Target Tokens: 8,500,000,000 (8.5 Billion tokens)
# - Global Batch: 32 Seqs x 2048 Context = 65,536 Tokens/step (4 seqs/core safe for 16GB HBM)
# - Primary Optimizer: Hybrid Muon + AdamW (switchable to adamw or muon)
# - Engine Optimizations:
#   1. steps_per_execution=64 (4.19M tokens per TPU-host round-trip)
#   2. Precomputed RoPE tables pre-cast to tf.bfloat16 at initialization
#   3. Static context length T=2048 with -1e4 bfloat16-safe causal mask
#   4. Tile + reshape GQA KV expansion (avoids tf.repeat intermediates)
#   5. Distributed multi-core C++ binary shard reader with automatic discovery
#   6. True multi-session resumption: step offset, shard skipping, LR schedule alignment
#   7. Session Safety: 8.4-Hour Watchdog callback + periodic weights export + optional HF Hub sync
# ==============================================================================

import os, json, sys, time, math, json, gc
import numpy as np
import tensorflow as tf

print(f"TensorFlow: {tf.__version__} | Python: {sys.version.split()[0]}")
tf.keras.mixed_precision.set_global_policy("mixed_bfloat16")
print(f"Global Precision Policy: {tf.keras.mixed_precision.global_policy().name}")

# ── TPU initialization: FAIL HARD if unavailable ──
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
    print(f"TPU master: {tpu.master()}")

    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)

    strategy = tf.distribute.TPUStrategy(tpu)

except Exception as e:
    raise RuntimeError(
        "\n"
        "============================================================\n"
        "TPU INITIALIZATION FAILED\n"
        "This training run is TPU-only; CPU fallback is disabled.\n"
        f"Original error: {e}\n"
        "============================================================"
    ) from e

# Verify actual TPU devices
tpu_devices = tf.config.list_logical_devices("TPU")

if not tpu_devices:
    raise RuntimeError(
        "TPUStrategy was created, but no TPU devices are visible."
    )

print(f"✅ TPU active")
print(f"   Replicas: {strategy.num_replicas_in_sync}")
print(f"   TPU devices: {len(tpu_devices)}")

if strategy.num_replicas_in_sync != 8:
    raise RuntimeError(
        f"Expected TPU v5e-8 (8 replicas), "
        f"but detected {strategy.num_replicas_in_sync} replicas."
    )

# ── Architecture Constants (Alpha 450M) ──
SEQUENCE_LENGTH    = 2048
VOCAB_SIZE         = 48000
HIDDEN_SIZE        = 1280
INTERMEDIATE_SIZE  = 3456
NUM_LAYERS         = 22
NUM_HEADS          = 20
NUM_KV_HEADS       = 5
HEAD_DIM           = HIDDEN_SIZE // NUM_HEADS   # 64
GQA_REP            = NUM_HEADS // NUM_KV_HEADS  # 4
ROPE_THETA         = 100000.0
RMS_NORM_EPS       = 1e-6

# ── Training & Batching Constants ──
PER_REPLICA_BATCH  = 4                                                 # Safe for 16GB HBM per core
GLOBAL_BATCH_SIZE  = PER_REPLICA_BATCH * strategy.num_replicas_in_sync  # 32 seqs on 8 cores
TOKENS_PER_STEP    = GLOBAL_BATCH_SIZE * SEQUENCE_LENGTH               # 65,536 tokens/step
STEPS_PER_EXEC     = 64                                                # Matches alpha_450m.json
TARGET_TOKENS      = 8_500_000_000                                     # 8.5 Billion tokens
TOTAL_STEPS        = TARGET_TOKENS // TOKENS_PER_STEP                  # ~129,700 steps
WARMUP_STEPS       = 2000                                              # Scaled for batch size 32

# ── Optimizer Selection ──
SELECTED_OPTIMIZER = "hybrid_muon_adamw"

# ── Storage & Checkpointing ──
WORKING_DIR         = "/kaggle/working"
CHECKPOINT_DIR      = os.path.join(WORKING_DIR, "checkpoints")
CHECKPOINT_OUTPUT   = os.path.join(WORKING_DIR, "alpha_latest.weights.h5")
METADATA_OUTPUT     = os.path.join(WORKING_DIR, "checkpoint_metadata.json")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Dynamic Shard Directory Discovery ──
def discover_shard_dir():
    candidates = [
        "/kaggle/input/alpha-8b-shards",
        "/kaggle/input/alpha-shards",
        "/kaggle/input/alpha-pretraining-shards",
        "/kaggle/input",
    ]
    for c in candidates:
        if os.path.exists(c):
            for root, _, files in os.walk(c):
                if any(f.endswith(".bin") for f in files):
                    return root
    return "/kaggle/input/alpha-8b-shards"

SHARD_DIR = discover_shard_dir()

# ── Dynamic Prior Checkpoint Discovery ──
def discover_prior_checkpoint():
    found_ckpts = []
    # Search /kaggle/input for mounted previous run outputs
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                if f.endswith(".weights.h5"):
                    meta_path = os.path.join(root, "checkpoint_metadata.json")
                    meta = {}
                    if os.path.exists(meta_path):
                        try:
                            with open(meta_path) as mf:
                                meta = json.load(mf)
                        except Exception:
                            pass
                    step = meta.get("step", 0)
                    found_ckpts.append((step, os.path.join(root, f), meta_path if os.path.exists(meta_path) else None))
    # Search /kaggle/working
    if os.path.exists(WORKING_DIR):
        for root, _, files in os.walk(WORKING_DIR):
            for f in files:
                if f.endswith(".weights.h5"):
                    meta_path = os.path.join(root, "checkpoint_metadata.json")
                    meta = {}
                    if os.path.exists(meta_path):
                        try:
                            with open(meta_path) as mf:
                                meta = json.load(mf)
                        except Exception:
                            pass
                    step = meta.get("step", 0)
                    found_ckpts.append((step, os.path.join(root, f), meta_path if os.path.exists(meta_path) else None))
    if found_ckpts:
        found_ckpts.sort(key=lambda x: x[0], reverse=True)
        return found_ckpts[0]
    return (0, None, None)

INITIAL_STEP, PRIOR_WEIGHTS, PRIOR_META = discover_prior_checkpoint()

print(f"Global Batch: {GLOBAL_BATCH_SIZE} seqs | {TOKENS_PER_STEP:,} tokens/step")
print(f"Target Budget: {TARGET_TOKENS:,} tokens = {TOTAL_STEPS:,} total steps")
print(f"XLA Execution: steps_per_execution={STEPS_PER_EXEC} ({STEPS_PER_EXEC * TOKENS_PER_STEP:,} tokens/exec)")
print(f"Active Optimizer: {SELECTED_OPTIMIZER}")
print(f"Shard Directory: {SHARD_DIR}")
if PRIOR_WEIGHTS:
    print(f"🔄 Detected Previous Checkpoint: {PRIOR_WEIGHTS} at step {INITIAL_STEP:,}")
else:
    print("🆕 Clean pretraining run from Step 0.")


In [ ]:
# ==============================================================================
# CELL 02 — MODEL ARCHITECTURE (Static Shapes, Pre-Cast RoPE, Tile-Reshape GQA)
# ==============================================================================

from tensorflow.keras import layers, models


class KerasRMSNorm(layers.Layer):
    """Root Mean Square Layer Normalization with learnable scaling."""
    def __init__(self, dim, eps=RMS_NORM_EPS, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps
        self.dim = dim

    def build(self, input_shape):
        self.weight = self.add_weight(shape=(self.dim,), name="weight", initializer="ones")

    def call(self, x):
        variance = tf.reduce_mean(tf.square(tf.cast(x, tf.float32)), axis=-1, keepdims=True)
        normed = tf.cast(x, tf.float32) * tf.math.rsqrt(variance + tf.cast(self.eps, tf.float32))
        return tf.cast(normed * tf.cast(self.weight, tf.float32), x.dtype)

    def get_config(self):
        config = super().get_config()
        config.update({"dim": self.dim, "eps": self.eps})
        return config


class PrecomputedRoPE(layers.Layer):
    """Static Rotary Position Embedding tables pre-cast to bfloat16 at __init__.
    Eliminates per-call casting overhead and avoids dynamic dimension queries."""
    def __init__(self, head_dim=HEAD_DIM, max_ctx=SEQUENCE_LENGTH, theta=ROPE_THETA, **kwargs):
        super().__init__(**kwargs)
        self.head_dim = head_dim
        indices = tf.range(0, head_dim, 2, dtype=tf.float32)
        inv_freq = 1.0 / (theta ** (indices / head_dim))
        t = tf.range(max_ctx, dtype=tf.float32)
        freqs = tf.einsum("i,j->ij", t, inv_freq)
        cos_tab = tf.concat([tf.cos(freqs), tf.cos(freqs)], axis=-1)
        sin_tab = tf.concat([tf.sin(freqs), tf.sin(freqs)], axis=-1)
        # Pre-cast to bfloat16 + add [1, 1, T, D] broadcast dimensions
        self.cos = tf.cast(cos_tab[tf.newaxis, tf.newaxis, :, :], tf.bfloat16)
        self.sin = tf.cast(sin_tab[tf.newaxis, tf.newaxis, :, :], tf.bfloat16)

    def call(self, x):
        d = self.head_dim
        x1, x2 = x[..., :d // 2], x[..., d // 2:]
        cos = tf.cast(self.cos, x.dtype)
        sin = tf.cast(self.sin, x.dtype)
        return (x * cos) + (tf.concat([-x2, x1], axis=-1) * sin)


class OptimizedGQA(layers.Layer):
    """Grouped-Query Attention (4:1) with QK-Norm, static causal mask, and tile+reshape KV expansion."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.q_proj = layers.Dense(NUM_HEADS * HEAD_DIM, use_bias=False, name="q_proj")
        self.k_proj = layers.Dense(NUM_KV_HEADS * HEAD_DIM, use_bias=False, name="k_proj")
        self.v_proj = layers.Dense(NUM_KV_HEADS * HEAD_DIM, use_bias=False, name="v_proj")
        self.o_proj = layers.Dense(HIDDEN_SIZE, use_bias=False, name="o_proj")
        self.q_norm = KerasRMSNorm(HEAD_DIM, name="q_norm")
        self.k_norm = KerasRMSNorm(HEAD_DIM, name="k_norm")
        self.rope = PrecomputedRoPE()
        # Float32 band_part then cast to bool with [1, 1, T, T] shape for clean broadcasting
        float_mask = tf.linalg.band_part(tf.ones((SEQUENCE_LENGTH, SEQUENCE_LENGTH), dtype=tf.float32), -1, 0)
        self.causal_mask = tf.cast(float_mask, tf.bool)[tf.newaxis, tf.newaxis, :, :]
        self.scale = tf.cast(HEAD_DIM ** -0.5, tf.bfloat16)

    def call(self, x):
        B = tf.shape(x)[0]
        T = SEQUENCE_LENGTH  # Static compile-time constant
        q = tf.reshape(self.q_proj(x), (B, T, NUM_HEADS, HEAD_DIM))
        k = tf.reshape(self.k_proj(x), (B, T, NUM_KV_HEADS, HEAD_DIM))
        v = tf.reshape(self.v_proj(x), (B, T, NUM_KV_HEADS, HEAD_DIM))
        # Apply QK-Norm and RoPE; transpose to [B, H, T, D]
        q = self.rope(tf.transpose(self.q_norm(q), [0, 2, 1, 3]))
        k = self.rope(tf.transpose(self.k_norm(k), [0, 2, 1, 3]))
        v = tf.transpose(v, [0, 2, 1, 3])
        # GQA KV expansion via tile + reshape (XLA-friendly, eliminates tf.repeat intermediates)
        k = tf.reshape(
            tf.tile(k[:, :, tf.newaxis, :, :], [1, 1, GQA_REP, 1, 1]),
            (B, NUM_HEADS, T, HEAD_DIM)
        )
        v = tf.reshape(
            tf.tile(v[:, :, tf.newaxis, :, :], [1, 1, GQA_REP, 1, 1]),
            (B, NUM_HEADS, T, HEAD_DIM)
        )
        # Attention computation with bfloat16-safe -1e4 mask fill
        scores = tf.matmul(q, k, transpose_b=True) * tf.cast(self.scale, q.dtype)
        scores = tf.where(self.causal_mask, scores, tf.cast(-1e4, scores.dtype))
        attn = tf.nn.softmax(scores, axis=-1)
        out = tf.matmul(attn, v)
        out = tf.reshape(tf.transpose(out, [0, 2, 1, 3]), (B, T, HIDDEN_SIZE))
        return self.o_proj(out)


class TransformerBlock(layers.Layer):
    """Pre-norm Transformer block with GQA and SwiGLU FFN."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.attn_norm = KerasRMSNorm(HIDDEN_SIZE, name="attn_norm")
        self.attn = OptimizedGQA(name="self_attn")
        self.ffn_norm = KerasRMSNorm(HIDDEN_SIZE, name="ffn_norm")
        self.gate = layers.Dense(INTERMEDIATE_SIZE, use_bias=False, name="gate_proj")
        self.up = layers.Dense(INTERMEDIATE_SIZE, use_bias=False, name="up_proj")
        self.down = layers.Dense(HIDDEN_SIZE, use_bias=False, name="down_proj")

    def call(self, x):
        x = x + self.attn(self.attn_norm(x))
        h = self.ffn_norm(x)
        x = x + self.down(tf.nn.silu(self.gate(h)) * self.up(h))
        return x


def build_alpha_model():
    """Builds the Alpha 450M model with tied token embeddings and float32 logits."""
    inp = layers.Input(shape=(SEQUENCE_LENGTH,), dtype=tf.int32, name="input_ids")
    emb = layers.Embedding(VOCAB_SIZE, HIDDEN_SIZE, name="tok_embeddings")
    x = emb(inp)
    for i in range(NUM_LAYERS):
        x = TransformerBlock(name=f"block_{i:02d}")(x)
    x = KerasRMSNorm(HIDDEN_SIZE, name="final_norm")(x)
    # Project to float32 logits for full numerical stability during cross-entropy computation
    logits = layers.Lambda(
        lambda h: tf.matmul(
            tf.cast(h, tf.float32),
            tf.cast(getattr(emb, 'embeddings', emb.weights[0] if emb.weights else emb.embeddings), tf.float32),
            transpose_b=True
        ),
        name="logits"
    )(x)
    return models.Model(inputs=inp, outputs=logits, name="Alpha_450M_Master")

print("✅ Architecture layers and model builder compiled successfully.")


In [ ]:
# ==============================================================================
# CELL 03 — OPTIMIZER ENGINE (Newton-Schulz, HybridMuonAdamW, PureMuon, LR Schedule)
# ==============================================================================

def zeropower_via_newtonschulz5(G, steps=5, eps=1e-7):
    """Compute G @ (G^T G)^{-1/2} via 5th-order Newton-Schulz iteration on TPU MXUs.
    Uses explicit bfloat16 constants to prevent TensorFlow dtype mismatch errors."""
    a = tf.cast(3.4445, tf.bfloat16)
    b = tf.cast(-4.7750, tf.bfloat16)
    c = tf.cast(2.0315, tf.bfloat16)
    eps_val = tf.cast(eps, tf.bfloat16)

    X = tf.cast(G, tf.bfloat16)
    X = X / (tf.norm(X) + eps_val)
    rows, cols = int(X.shape[0]), int(X.shape[1])  # Guaranteed Python ints
    transposed = False
    if rows > cols:
        X = tf.transpose(X)
        transposed = True
    for _ in range(steps):
        A = tf.matmul(X, X, transpose_b=True)
        B = b * A + c * tf.matmul(A, A)
        X = a * X + tf.matmul(B, X)
    if transposed:
        X = tf.transpose(X)
    return tf.cast(X, G.dtype)


class CosineDecayWithWarmup(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Cosine decay with linear warmup matching standard LLM pretraining curves."""
    def __init__(self, base_lr, warmup_steps=WARMUP_STEPS, total_steps=TOTAL_STEPS, min_lr_ratio=0.1):
        super().__init__()
        self.base_lr = float(base_lr)
        self.warmup_steps = float(warmup_steps)
        self.total_steps = float(total_steps)
        self.min_lr_ratio = float(min_lr_ratio)

    def __call__(self, step):
        step_f = tf.cast(step, tf.float32)
        warmup_lr = self.base_lr * (step_f / tf.maximum(self.warmup_steps, 1.0))
        progress = (step_f - self.warmup_steps) / tf.maximum(self.total_steps - self.warmup_steps, 1.0)
        progress = tf.clip_by_value(progress, 0.0, 1.0)
        cosine_factor = 0.5 * (1.0 + tf.cos(math.pi * progress))
        decayed_lr = self.base_lr * (self.min_lr_ratio + (1.0 - self.min_lr_ratio) * cosine_factor)
        return tf.where(step_f < self.warmup_steps, warmup_lr, decayed_lr)

    def get_config(self):
        return {"base_lr": self.base_lr, "warmup_steps": self.warmup_steps,
                "total_steps": self.total_steps, "min_lr_ratio": self.min_lr_ratio}


class HybridMuonAdamW(tf.keras.optimizers.Optimizer):
    """Production Hybrid Optimizer with decoupled weight decay and group-specific LR schedules."""
    def __init__(self, lr_muon=0.02, lr_adam=5e-4, muon_momentum=0.95,
                 beta1=0.9, beta2=0.95, weight_decay=0.01, ns_steps=5,
                 clipnorm=1.0, **kwargs):
        super().__init__(learning_rate=lr_muon, clipnorm=clipnorm, **kwargs)
        self.lr_muon = lr_muon
        self.lr_adam = lr_adam
        self.muon_momentum = muon_momentum
        self.beta1 = beta1
        self.beta2 = beta2
        self.weight_decay = weight_decay
        self.ns_steps = ns_steps

    def build(self, var_list):
        super().build(var_list)
        # Type-selective allocation: 2D matrices get muon_buf only; 1D/embeddings get adam_m/v only.
        # Saves ~3GB HBM across 444M parameters on 16GB TPU v5e cores.
        self.muon_bufs = [
            self.add_variable_from_reference(v, name="muon_buf") if self._is_muon_param(v) else None
            for v in var_list
        ]
        self.adam_m = [
            self.add_variable_from_reference(v, name="adam_m") if not self._is_muon_param(v) else None
            for v in var_list
        ]
        self.adam_v = [
            self.add_variable_from_reference(v, name="adam_v") if not self._is_muon_param(v) else None
            for v in var_list
        ]

    def _is_muon_param(self, var):
        """2D hidden weight matrices -> Muon; 1D, norms, and embeddings -> AdamW."""
        return len(var.shape) == 2 and "tok" not in var.name

    def _get_lr(self, lr_attr):
        if callable(lr_attr):
            return tf.cast(lr_attr(self.iterations), tf.float32)
        return tf.cast(lr_attr, tf.float32)

    def update_step(self, gradient, variable, learning_rate):
        if isinstance(gradient, tf.IndexedSlices):
            gradient = tf.convert_to_tensor(gradient)

        idx = self._get_variable_index(variable)
        cur_lr_muon = self._get_lr(self.lr_muon)
        cur_lr_adam = self._get_lr(self.lr_adam)

        if self._is_muon_param(variable):
            if self.weight_decay > 0:
                variable.assign(variable * tf.cast(1.0 - cur_lr_muon * self.weight_decay, variable.dtype))
            buf = self.muon_bufs[idx]
            buf.assign(self.muon_momentum * buf + gradient)
            update = gradient + self.muon_momentum * buf
            scale = tf.sqrt(tf.constant(float(max(variable.shape)), dtype=tf.float32))
            ortho = zeropower_via_newtonschulz5(update, steps=self.ns_steps)
            step_update = tf.cast(cur_lr_muon * scale, ortho.dtype) * ortho
            variable.assign_sub(tf.cast(step_update, variable.dtype))
        else:
            if self.weight_decay > 0:
                variable.assign(variable * tf.cast(1.0 - cur_lr_adam * self.weight_decay, variable.dtype))
            t = tf.cast(self.iterations + 1, tf.float32)
            m = self.adam_m[idx]
            v = self.adam_v[idx]
            m.assign(self.beta1 * m + (1.0 - self.beta1) * gradient)
            v.assign(self.beta2 * v + (1.0 - self.beta2) * tf.square(gradient))
            m_hat = m / (1.0 - tf.pow(self.beta1, t))
            v_hat = v / (1.0 - tf.pow(self.beta2, t))
            step_update = cur_lr_adam * m_hat / (tf.sqrt(v_hat) + 1e-8)
            variable.assign_sub(tf.cast(step_update, variable.dtype))

    def get_config(self):
        config = super().get_config()
        config.update({"lr_muon": self.lr_muon, "lr_adam": self.lr_adam,
                       "muon_momentum": self.muon_momentum, "beta1": self.beta1,
                       "beta2": self.beta2, "weight_decay": self.weight_decay,
                       "ns_steps": self.ns_steps})
        return config


class PureMuon(tf.keras.optimizers.Optimizer):
    """Pure Muon: Newton-Schulz for 2D hidden weights, Nesterov SGD for 1D/embeddings."""
    def __init__(self, lr_muon=0.02, lr_1d=5e-4, momentum=0.95,
                 weight_decay=0.01, ns_steps=5, clipnorm=1.0, **kwargs):
        super().__init__(learning_rate=lr_muon, clipnorm=clipnorm, **kwargs)
        self.lr_muon = lr_muon
        self.lr_1d = lr_1d
        self.momentum_val = momentum
        self.weight_decay = weight_decay
        self.ns_steps = ns_steps

    def build(self, var_list):
        super().build(var_list)
        self.bufs = [self.add_variable_from_reference(v, name="buf") for v in var_list]

    def _get_lr(self, lr_attr):
        if callable(lr_attr):
            return tf.cast(lr_attr(self.iterations), tf.float32)
        return tf.cast(lr_attr, tf.float32)

    def update_step(self, gradient, variable, learning_rate):
        if isinstance(gradient, tf.IndexedSlices):
            gradient = tf.convert_to_tensor(gradient)

        idx = self._get_variable_index(variable)
        is_2d = len(variable.shape) == 2 and "tok" not in variable.name
        cur_lr_muon = self._get_lr(self.lr_muon)
        cur_lr_1d = self._get_lr(self.lr_1d)
        wd_lr = cur_lr_muon if is_2d else cur_lr_1d
        if self.weight_decay > 0:
            variable.assign(variable * tf.cast(1.0 - wd_lr * self.weight_decay, variable.dtype))
        buf = self.bufs[idx]
        buf.assign(self.momentum_val * buf + gradient)
        update = gradient + self.momentum_val * buf
        if is_2d:
            scale = tf.sqrt(tf.constant(float(max(variable.shape)), dtype=tf.float32))
            ortho = zeropower_via_newtonschulz5(update, steps=self.ns_steps)
            step_update = tf.cast(cur_lr_muon * scale, ortho.dtype) * ortho
            variable.assign_sub(tf.cast(step_update, variable.dtype))
        else:
            variable.assign_sub(tf.cast(cur_lr_1d * update, variable.dtype))

    def get_config(self):
        config = super().get_config()
        config.update({"lr_muon": self.lr_muon, "lr_1d": self.lr_1d,
                       "momentum": self.momentum_val, "weight_decay": self.weight_decay,
                       "ns_steps": self.ns_steps})
        return config


def create_optimizer(mode=SELECTED_OPTIMIZER, use_schedules=True):
    """Creates optimizer with warmup + cosine decay matching alpha_450m.json config."""
    if use_schedules:
        muon_lr = CosineDecayWithWarmup(base_lr=0.02, warmup_steps=WARMUP_STEPS, total_steps=TOTAL_STEPS)
        adam_lr = CosineDecayWithWarmup(base_lr=5e-4, warmup_steps=WARMUP_STEPS, total_steps=TOTAL_STEPS)
    else:
        muon_lr, adam_lr = 0.02, 5e-4

    if mode == "hybrid_muon_adamw":
        return HybridMuonAdamW(
            lr_muon=muon_lr, lr_adam=adam_lr, muon_momentum=0.95,
            beta1=0.9, beta2=0.95, weight_decay=0.01, ns_steps=5, clipnorm=1.0
        )
    elif mode == "adamw":
        return tf.keras.optimizers.AdamW(
            learning_rate=adam_lr, beta_1=0.9, beta_2=0.95,
            weight_decay=0.01, epsilon=1e-8, clipnorm=1.0
        )
    elif mode == "muon":
        return PureMuon(
            lr_muon=muon_lr, lr_1d=adam_lr, momentum=0.95,
            weight_decay=0.01, ns_steps=5, clipnorm=1.0
        )
    else:
        raise ValueError(f"Unknown optimizer configuration: {mode}")

print(f"✅ Optimizer engine ready. Configured primary candidate: '{SELECTED_OPTIMIZER}' with Cosine LR schedule.")


In [ ]:
# ==============================================================================
# CELL 04 — HIGH-SPEED DISTRIBUTED SHARD PIPELINE (C++ FixedLengthRecordDataset)
# ==============================================================================

def build_training_dataset(strategy, shard_dir=SHARD_DIR, initial_step=INITIAL_STEP):
    """Builds distributed multi-core dataset pipeline via distribute_datasets_from_function.
    Uses tf.data.FixedLengthRecordDataset for zero-copy C++ I/O directly from uint16 binary shards.
    Automatically skips records already consumed in previous sessions to prevent duplicate replay."""
    shard_files = sorted([
        os.path.join(shard_dir, f) for f in os.listdir(shard_dir)
        if f.endswith(".bin") and "shard" in f
    ]) if os.path.exists(shard_dir) else []

    record_bytes = (SEQUENCE_LENGTH + 1) * 2

    def dataset_fn(input_context):
        per_replica_batch = input_context.get_per_replica_batch_size(GLOBAL_BATCH_SIZE)
        
        if shard_files:
            pipe_id = input_context.input_pipeline_id
            num_pipes = input_context.num_input_pipelines
            assigned = [sf for idx, sf in enumerate(shard_files) if idx % num_pipes == pipe_id]
            if not assigned:
                assigned = shard_files

            ds = tf.data.FixedLengthRecordDataset(assigned, record_bytes=record_bytes)
            
            def parse_binary_record(raw):
                tokens = tf.io.decode_raw(raw, tf.uint16)
                tokens = tf.cast(tokens, tf.int32)
                return tokens[:-1], tokens[1:]

            ds = ds.map(parse_binary_record, num_parallel_calls=tf.data.AUTOTUNE)
            ds = ds.repeat()
            ds = ds.shuffle(buffer_size=2048, seed=42)

            # Skip records already trained in previous sessions to prevent token replay
            if initial_step > 0:
                records_to_skip = initial_step * per_replica_batch
                ds = ds.skip(records_to_skip)
        else:
            ds = tf.data.Dataset.range(1).repeat()
            ds = ds.map(
                lambda _: (
                    tf.random.uniform((SEQUENCE_LENGTH,), 0, VOCAB_SIZE, dtype=tf.int32),
                    tf.random.uniform((SEQUENCE_LENGTH,), 0, VOCAB_SIZE, dtype=tf.int32),
                ),
                num_parallel_calls=tf.data.AUTOTUNE,
            )

        ds = ds.batch(per_replica_batch, drop_remainder=True)
        ds = ds.prefetch(tf.data.AUTOTUNE)
        return ds

    if shard_files:
        print(f"✅ Shard Pipeline: Found {len(shard_files)} binary shards in {shard_dir}")
        if initial_step > 0:
            print(f"⏩ Resuming pipeline: Skipping {initial_step * GLOBAL_BATCH_SIZE:,} previously trained records.")
    else:
        print(f"⚠️ Shards not detected in {shard_dir}. Using synthetic dataset stream for dry-run verification.")

    return strategy.distribute_datasets_from_function(dataset_fn)

train_ds = build_training_dataset(strategy, shard_dir=SHARD_DIR, initial_step=INITIAL_STEP)
print(f"✅ Distributed dataset configured with static per-replica batch size: {PER_REPLICA_BATCH}")


In [ ]:
# ==============================================================================
# CELL 05 — WATCHDOGS, MODEL COMPILATION, CHECKPOINT RESUME & PRETRAINING LOOP
# ==============================================================================

def push_to_hf_hub_if_configured(weights_path, metadata_path, repo_id=None):
    """Optionally uploads checkpoint to private Hugging Face repo if HF_TOKEN is configured."""
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        return  # Gracefully skip if secret is not set
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=token)
        if repo_id is None:
            user_info = api.whoami()
            repo_id = f"{user_info['name']}/alpha-450m-checkpoints"
        api.create_repo(repo_id=repo_id, exist_ok=True, private=True)
        api.upload_file(path_or_fileobj=weights_path, path_in_repo="alpha_latest.weights.h5", repo_id=repo_id)
        if os.path.exists(metadata_path):
            api.upload_file(path_or_fileobj=metadata_path, path_in_repo="checkpoint_metadata.json", repo_id=repo_id)
        print(f"☁️ [HF HUB SYNC] Checkpoint successfully backed up to https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"⚠️ [HF HUB SYNC] Note: Cloud sync skipped ({e}). Local checkpoint is secure.")


class SessionWatchdog(tf.keras.callbacks.Callback):
    """Monitors training duration and cleanly stops at 8.4 hours to ensure
    unhurried weights export before Kaggle's 9.0-hour hard session cutoff."""
    def __init__(self, max_hours=8.4):
        super().__init__()
        self.start_time = time.time()
        self.max_seconds = max_hours * 3600.0

    def on_train_batch_end(self, batch, logs=None):
        elapsed = time.time() - self.start_time
        if elapsed >= self.max_seconds:
            print(f"\n⏰ [WATCHDOG] 8.4h session ceiling reached ({elapsed/3600:.2f}h elapsed).")
            print("   Cleanly stopping training loop for final checkpoint export...")
            self.model.stop_training = True


class PeriodicWeightsCheckpoint(tf.keras.callbacks.Callback):
    """Periodically exports model weights and metadata every save_freq batches."""
    def __init__(self, weights_path, metadata_path, initial_step=0, save_freq=128):
        super().__init__()
        self.weights_path = weights_path
        self.metadata_path = metadata_path
        self.initial_step = initial_step
        self.save_freq = save_freq

    def on_train_batch_end(self, batch, logs=None):
        current_step = self.initial_step + batch + 1
        if current_step % self.save_freq == 0:
            self.model.save_weights(self.weights_path)
            loss_val = float(logs.get('loss', 0.0)) if logs else 0.0
            meta = {
                "step": current_step,
                "tokens_seen": current_step * TOKENS_PER_STEP,
                "total_steps": TOTAL_STEPS,
                "progress_pct": round((current_step / TOTAL_STEPS) * 100, 2),
                "loss": loss_val,
                "timestamp": time.time(),
                "optimizer": SELECTED_OPTIMIZER
            }
            with open(self.metadata_path, "w") as f:
                json.dump(meta, f, indent=2)
            print(f"\n💾 [CHECKPOINT] Step {current_step:,}/{TOTAL_STEPS:,} ({meta['progress_pct']}%) | "
                  f"Loss: {loss_val:.4f} | Tokens: {meta['tokens_seen']:,}")


# ── Compile Model within TPUStrategy Scope ──
with strategy.scope():
    model = build_alpha_model()
    optimizer = create_optimizer(SELECTED_OPTIMIZER, use_schedules=True)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(optimizer=optimizer, loss=loss_fn, steps_per_execution=STEPS_PER_EXEC)

param_count = model.count_params()
print(f"✅ Model compiled with {SELECTED_OPTIMIZER.upper()} inside TPUStrategy.")
print(f"   Total Parameters: {param_count:,} | steps_per_execution: {STEPS_PER_EXEC}")

# ── Resume Logic ──
if PRIOR_WEIGHTS and os.path.exists(PRIOR_WEIGHTS):
    print(f"🔄 Restoring weights from previous checkpoint: {PRIOR_WEIGHTS}")
    model.load_weights(PRIOR_WEIGHTS)
    # Synchronize optimizer step counter so CosineDecay schedule resumes at the exact step!
    optimizer.iterations.assign(INITIAL_STEP)
    print(f"✅ Resumed successfully from Step {INITIAL_STEP:,}. Optimizer iterations set to {INITIAL_STEP:,}.")
else:
    print("🆕 Initializing Alpha 450M from scratch for pretraining run.")

# ── Launch Pretraining Loop ──
REMAINING_STEPS = TOTAL_STEPS - INITIAL_STEP

if REMAINING_STEPS <= 0:
    print(f"🎉 Pretraining target of {TARGET_TOKENS:,} tokens has already been achieved! ({INITIAL_STEP:,} steps completed)")
else:
    callbacks = [
        SessionWatchdog(max_hours=8.4),
        PeriodicWeightsCheckpoint(CHECKPOINT_OUTPUT, METADATA_OUTPUT, initial_step=INITIAL_STEP, save_freq=128)
    ]

    print(f"\n🚀 Launching Alpha 450M Master Pretraining on TPU v5e-8...")
    print(f"   Initial Step: {INITIAL_STEP:,} | Remaining Steps: {REMAINING_STEPS:,} | Total Goal: {TOTAL_STEPS:,}")
    print(f"   Global Batch: {GLOBAL_BATCH_SIZE} | Tokens/step: {TOKENS_PER_STEP:,}")

    start_train_time = time.time()
    history = model.fit(train_ds, steps_per_epoch=REMAINING_STEPS, epochs=1, callbacks=callbacks, verbose=1)
    total_elapsed = time.time() - start_train_time

    # Save final weights and metadata
    final_step = INITIAL_STEP + len(history.history.get('loss', [])) * STEPS_PER_EXEC
    if final_step > TOTAL_STEPS:
        final_step = TOTAL_STEPS
    model.save_weights(CHECKPOINT_OUTPUT)
    final_meta = {
        "step": final_step,
        "tokens_seen": final_step * TOKENS_PER_STEP,
        "total_steps": TOTAL_STEPS,
        "progress_pct": round((final_step / TOTAL_STEPS) * 100, 2),
        "loss": float(history.history.get('loss', [-1.0])[-1]) if history.history.get('loss') else 0.0,
        "timestamp": time.time(),
        "optimizer": SELECTED_OPTIMIZER
    }
    with open(METADATA_OUTPUT, "w") as f:
        json.dump(final_meta, f, indent=2)

    print(f"\n🎉 Pretraining session complete!")
    print(f"   Elapsed time: {total_elapsed / 3600:.2f} hours")
    print(f"   Final Step: {final_step:,}/{TOTAL_STEPS:,} ({final_meta['progress_pct']}%)")
    print(f"   Saved weights to: {CHECKPOINT_OUTPUT}")

    # Optional automated cloud backup to Hugging Face
    push_to_hf_hub_if_configured(CHECKPOINT_OUTPUT, METADATA_OUTPUT)
